In [ ]:
from datetime import datetime
from pathlib import Path
from typing import Tuple, Dict

from minbpe import RegexTokenizer as Tokenizer
from transformer.model import GPTLanguageModel

import matplotlib.pyplot as plt
import numpy as np
import torch
torch.manual_seed(3647)
torch.set_float32_matmul_precision('high')
from tqdm import tqdm

In [ ]:
def now():
    return datetime.now().astimezone().strftime('%FT%T%:z')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
data_dir = Path("data") / "ch08"

In [ ]:
tokenizer = Tokenizer()
tokenizer.load(model_file=data_dir / "darija_tokenizer.model")

data = np.load(data_dir / "encoded_atlaset.npy", mmap_mode='r')
print('Data shape:', data.shape)

In [ ]:
parameters = {
  "block_size": 1024,
  "n_embd": 512,
  "n_head": 8,
  "n_layer": 8,
  "dropout": 0.2,
  "vocab_size": len(tokenizer.vocab),
}

split_index = int(0.9*len(data))
batch_size = 4
learning_rate = 3e-4
gradient_accumulation_steps = 8
eval_interval = 3000
save_interval = 10000

model = GPTLanguageModel(
    vocab_size=parameters['vocab_size'],
    block_size=parameters['block_size'],
    n_embd=parameters['n_embd'],
    n_head=parameters['n_head'],
    n_layer=parameters['n_layer'],
    dropout=parameters['dropout'],
    device=device
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
optimizer.zero_grad(set_to_none=True)

print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

In [ ]:
def get_batch_for_loss_estimation(split: str) -> Tuple[torch.Tensor, torch.Tensor]:
    start_index, end_index = (0, split_index) if split == 'train' else (split_index, len(data))

    available_blocks = (end_index - start_index - 1) // block_size
    block_indices = torch.randint(0, available_blocks, (batch_size,))

    x_batch, y_batch = [], []
    for i in block_indices:
        block_start = start_index + (i * block_size)
        x_batch.append(data[block_start:block_start+block_size])
        y_batch.append(data[block_start+1:block_start+block_size+1])

    x_batch = torch.tensor(np.array(x_batch), dtype=torch.long).to(device)
    y_batch = torch.tensor(np.array(y_batch), dtype=torch.long).to(device)

    return x_batch, y_batch


@torch.no_grad()
def estimate_loss() -> Dict:
    output = {}
    eval_iters = 1000
    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch_for_loss_estimation(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        output[split] = losses.mean()

    model.train()

    return output

def save_checkpoint(
    model: GPTLanguageModel,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    prefix: str,
) -> None:
    losses = estimate_loss()

    meta = {
        "created_at": now(),
        "parameters": parameters,
        "epoch": epoch,
        "train_loss": float(losses['train']),
        "val_loss": float(losses['val']),
    }

    checkpoint = {
        'meta': meta,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }

    with open(prefix + ".json", 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    torch.save(checkpoint, prefix + ".pt")

In [ ]:
# Calculate the number of complete non-overlapping blocks
non_overlapping_blocks = (split_index - 1) // block_size
total_batches = non_overlapping_blocks // batch_size

batches_processed = 0
train_losses, val_losses = [], []

for i in tqdm(
    iterable=range(0, non_overlapping_blocks, batch_size),
    desc="Processing",
    total=total_batches,
):
    # Load a batch of non-overlapping blocks
    x_batch, y_batch = [], []
    for j in range(batch_size):
        if i+j < non_overlapping_blocks:
            block_start = (i+j) * block_size
            x = data[block_start:block_start+block_size]
            y = data[block_start+1:block_start+block_size+1]
            x_batch.append(x)
            y_batch.append(y)

    if len(x_batch) == 0:
        continue

    x_batch = torch.tensor(np.array(x_batch), dtype=torch.long).to(device)
    y_batch = torch.tensor(np.array(y_batch), dtype=torch.long).to(device)

    # Forward pass
    logits, loss = model(x_batch, y_batch)
    loss /= gradient_accumulation_steps
    loss.backward()

    # Gradient accumulation
    batches_processed += 1
    if batches_processed % gradient_accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    # Evaluate the model
    if batches_processed % eval_interval == 0:
        losses = estimate_loss()
        print(
            f"Batch {batches_processed}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])

    # Save the model
    if batches_processed % save_interval == 0:
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=batches_processed,
            prefix=str(data_dir / "checkpoint_batch_{batches_processed:07_}"),
        )

if batches_processed % gradient_accumulation_steps != 0:
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

save_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=batches_processed,
    loss=loss.item(),
    prefix=str(data_dir / "checkpoint_batch_{batches_processed:07_}"),
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Evaluation Step")
plt.ylim(0)
plt.ylabel("Loss")
plt.title("Training and Validation Loss Over Time")
plt.legend()
plt.grid()
plt.show()

In [ ]:
input_tokens = tokenizer.encode("ماهي الأسباب الرئيسية اللي خلاو الفرنسيين")

input_tokens = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model.generate(input_tokens=input_tokens, max_new_tokens=100)

print(tokenizer.decode(output[0].tolist()))